In [1]:
import torch
import torch.nn as nn

In [2]:
class InceptionModule(nn.Module):
    def __init__(self, in_ch, out_1x1, out_3x3_reduce, out_3x3,
                              out_5x5_reduce, out_5x5, out_pool):
        super().__init__()
        # Nhánh 1: Conv 1×1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, out_1x1, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        # Nhánh 2: Conv 1×1 → Conv 3×3
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, out_3x3_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_3x3_reduce, out_3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        # Nhánh 3: Conv 1×1 → Conv 5×5
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, out_5x5_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_5x5_reduce, out_5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )
        # Nhánh 4: MaxPool 3×3 → Conv 1×1
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_ch, out_pool, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)  # Ghép theo chiều kênh

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        # Stem (phần đầu)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),  nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64, 64, kernel_size=1),                      nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),          nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
      
        self.inception3a = InceptionModule(192,  64,  96, 128, 16,  32,  32)  # out=256
        self.inception3b = InceptionModule(256, 128, 128, 192, 32,  96,  64)  # out=480
        self.maxpool3    = nn.MaxPool2d(3, stride=2, padding=1)
        self.inception4a = InceptionModule(480, 192,  96, 208, 16,  48,  64)  # out=512
        self.inception4b = InceptionModule(512, 160, 112, 224, 24,  64,  64)  # out=512
        self.inception4c = InceptionModule(512, 128, 128, 256, 24,  64,  64)  # out=512
        self.inception4d = InceptionModule(512, 112, 144, 288, 32,  64,  64)  # out=528
        self.inception4e = InceptionModule(528, 256, 160, 320, 32, 128, 128)  # out=832
        self.maxpool4    = nn.MaxPool2d(3, stride=2, padding=1)
        self.inception5a = InceptionModule(832, 256, 160, 320, 32, 128, 128)  # out=832
        self.inception5b = InceptionModule(832, 384, 192, 384, 48, 128, 128)  # out=1024
        self.avgpool    = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(0.4)
        self.fc         = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)
        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)
        x = self.inception5a(x)
        x = self.inception5b(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [4]:
model = GoogLeNet(num_classes=1000)
dummy = torch.randn(2, 3, 224, 224)
out = model(dummy)
print(f"Output shape: {out.shape}")    # (2, 1000)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,}")  

Output shape: torch.Size([2, 1000])
Total params: 6,998,552
